In [22]:
import pandas as pd

In [23]:
data = pd.read_csv('sales_data.csv', sep=';')
#cek data
data.head()

,product_id,product_name,jumlah_penjualan,harga,diskon,status,Unnamed: 6
0,P00001,Produk 1,57,26556,25,Tidak,NaN
1,P00002,Produk 2,140,84196,5,Laris,NaN
2,P00003,Produk 3,71,46868,25,Tidak,NaN
3,P00004,Produk 4,279,42790,20,Laris,NaN
4,P00005,Produk 5,216,28331,0,Laris,NaN


In [24]:
#drop colom tidak berguna
data = data.drop(columns=['Unnamed: 6'])

In [25]:
data.head()

,product_id,product_name,jumlah_penjualan,harga,diskon,status
0,P00001,Produk 1,57,26556,25,Tidak
1,P00002,Produk 2,140,84196,5,Laris
2,P00003,Produk 3,71,46868,25,Tidak
3,P00004,Produk 4,279,42790,20,Laris
4,P00005,Produk 5,216,28331,0,Laris


In [26]:
print(data.dtypes)

product_id            str
product_name          str
jumlah_penjualan    int64
harga               int64
diskon              int64
status                str
dtype: object


In [27]:
#cek data yang null
print(data.isnull().sum())

product_id          0
product_name        0
jumlah_penjualan    0
harga               0
diskon              0
status              0
dtype: int64


In [28]:
#import library yang diperlukanabs
#karena pandas sudah tidak perlu importab
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error

from sklearn.ensemble import RandomForestClassifier

In [29]:
#beri target dan features
target = 'status_numeric'
data['status_numeric'] = data['status'].map({'Laris': 1, 'Tidak': 0})
drop_cols = ['product_id', 'product_name', 'status']
                                            
X = data.drop(columns=drop_cols + ['status_numeric'])
y = data['status_numeric']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns
#bersihkan angka beri nilai tengah jika nan atau null
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
#berikan text jika status kosong beri yang paling banyak muncul
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [44]:
#mulai training data 
model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=10,
        max_depth=2,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1 
    ))
])

model.fit(X_train, y_train)

preds = model.predict(X_test)

#sekarang cek akurasi
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, preds)
print(f"Akurasi: {accuracy * 100}%")
print(classification_report(y_test, preds))

Akurasi: 94.6%
              precision    recall  f1-score   support

           0       1.00      0.81      0.90       286
           1       0.93      1.00      0.96       714

    accuracy                           0.95      1000
   macro avg       0.96      0.91      0.93      1000
weighted avg       0.95      0.95      0.94      1000



In [47]:
data_baru = pd.DataFrame({
    'jumlah_penjualan': [100],
    'harga': [50000],
    'diskon': [1]
})

hasil_prediksi = model.predict(data_baru)

status = "Laris" if hasil_prediksi[0] == 1 else "Tidak Laris"
print(f"Hasil Prediksi: {status}")

Hasil Prediksi: Laris


In [50]:
import joblib

joblib.dump(model, 'backend/model_final.pkl')
print("Model berhasil disimpan!")

Model berhasil disimpan!
